In [ ]:
import json
import pandas as pd
import numpy as np


In [ ]:
carparks = pd.read_csv("../data/HDBCarparkInformation_new.csv")

In [ ]:
carparks.head(5)

In [ ]:
carparks.shape

In [ ]:
def svy21_to_wgs84(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    a, f = 6378137.0, 1 / 298.257223563
    oLat, oLon, oN, oE, k = 1.366666, 103.833333, 38744.572, 28001.642, 1.0
    b = a * (1 - f)
    e2 = (2 * f) - (f ** 2)
    e4, e6 = e2 ** 2, e2 ** 3
    A0 = 1 - (e2/4) - (3*e4/64) - (5*e6/256)
    A2 = (3/8) * (e2 + (e4/4) + (15*e6/128))
    A4 = (15/256) * (e4 + (3*e6/4))
    A6 = (35/3072) * e6
    oLatRad = np.radians(oLat)
    M0 = a * ((A0*oLatRad) - (A2*np.sin(2*oLatRad)) + (A4*np.sin(4*oLatRad)) - (A6*np.sin(6*oLatRad)))
    Nn = y - oN
    M = M0 + (Nn/k)
    n = (a - b) / (a + b)
    G = a * (1-n) * (1-n**2) * (1 + (9*n**2/4) + (225*n**4/64)) * (np.pi/180)
    sigma = (M*np.pi) / (180*G)
    latPrime = (sigma + ((3*n/2) - (27*n**3/32))*np.sin(2*sigma) +
                ((21*n**2/16) - (55*n**4/32))*np.sin(4*sigma) +
                (151*n**3/96)*np.sin(6*sigma) + (1097*n**4/512)*np.sin(8*sigma))
    rho = (a*(1-e2)) / ((1 - e2*np.sin(latPrime)**2)**1.5)
    v = a / np.sqrt(1 - e2*np.sin(latPrime)**2)
    psi, t = v/rho, np.tan(latPrime)
    Ee = x - oE
    x1 = Ee / (k*v)
    x2, x3 = x1**2, x1**3
    x4, x5 = x2**2, x1*x2**2
    x6, x7 = x4*x2, x5*x2
    lat = (latPrime - (t/(k*rho))*(Ee*x1/2) +
           (t/(k*rho))*(Ee*x3/24)*(-4*psi**2 + 9*psi*(1-t**2) + 12*t**2) -
           (t/(k*rho))*(Ee*x5/720)*(8*psi**4*(11-24*t**2) - 12*psi**3*(21-71*t**2) +
            15*psi**2*(15-98*t**2+15*t**4) + 180*psi*(5*t**2-3*t**4) + 360*t**4) +
           (t/(k*rho))*(Ee*x7/40320)*(1385 + 3633*t**2 + 4095*t**4 + 1575*t**6))
    lon = (np.radians(oLon) + (x1/np.cos(latPrime)) -
           (x3/(6*np.cos(latPrime)))*(psi + 2*t**2) +
           (x5/(120*np.cos(latPrime)))*(-4*psi**3*(1-6*t**2) + psi**2*(9-68*t**2) +
            72*psi*t**2 + 24*t**4) -
           (x7/(5040*np.cos(latPrime)))*(61 + 662*t**2 + 1320*t**4 + 720*t**6))
    return np.degrees(lat), np.degrees(lon)


lat, lon = svy21_to_wgs84(carparks["x_coord"], carparks["y_coord"])
carparks["latitude"] = lat
carparks["longitude"] = lon
carparks.to_csv("../data/HDBCarparkInformation_new.csv", index=False)
carparks

In [ ]:
# check whether the coordinates fall within a reasonable Singapore geographic range
valid = (
    carparks["latitude"].between(1.1, 1.5)
    & carparks["longitude"].between(103.5, 104.1)
)

print("Valid coordinates:", valid.sum())
print("Invalid coordinates:", (~valid).sum())

In [ ]:
carparks["car_park_no"].is_unique

In [ ]:
carparks["longitude"].dtype

In [ ]:
# Step 1: keep the original raw values safe, untouched, before any casting
carparks["car_park_decks_raw"] = carparks["car_park_decks"]
carparks["gantry_height_raw"] = carparks["gantry_height"]

# Step 2: cast to numbers, turning anything unconvertible into NaN (not crashing)

carparks["car_park_decks"] = pd.to_numeric(carparks["car_park_decks"], errors="coerce")
carparks["gantry_height"] = pd.to_numeric(carparks["gantry_height"], errors="coerce")

# Step 3: check for anything that silently failed to convert
checks = {

    "car_park_decks": ("car_park_decks_raw", "car_park_decks"),
    "gantry_height": ("gantry_height_raw", "gantry_height"),
}

for field, (raw_col, new_col) in checks.items():
    failed = carparks[carparks[raw_col].notna() & carparks[new_col].isna()]
    print(f"{field}: {len(failed)} values failed to convert")
    if len(failed) > 0:
        print(failed[[raw_col]])

carparks.to_csv("../data/HDBCarparkInformation_new.csv", index=False)

In [ ]:
carparks.dtypes

In [ ]:

# Step 1: create the boolean flags
carparks["is_free_parking"] = ~carparks["free_parking"].str.upper().eq("NO")
carparks["has_short_term_parking"] = ~carparks["short_term_parking"].str.upper().eq("NO")

# Step 2: create the condition/detail columns
# if the value was "NO" or "YES" (plain answers), there's no extra condition to keep
# anything else (a time window or day restriction) gets kept as the condition
carparks["free_parking_condition"] = carparks["free_parking"].where(
    ~carparks["free_parking"].str.upper().isin(["NO", "YES", "WHOLE DAY"])
)
carparks["short_term_parking_condition"] = carparks["short_term_parking"].where(
    ~carparks["short_term_parking"].str.upper().isin(["NO", "YES", "WHOLE DAY"])
)

# Step 3: validation check -- make sure every row was classified, nothing silently lost
for col, raw_col in [("is_free_parking", "free_parking"), ("has_short_term_parking", "short_term_parking")]:
    missing = carparks[carparks[raw_col].notna() & carparks[col].isna()]
    print(f"{col}: {len(missing)} rows failed to classify")

carparks.to_csv("../data/HDBCarparkInformation_new.csv", index=False)

In [ ]:
# night_parking: sample values are "YES" / "NO"
# converting it to boolean
carparks["has_night_parking"] = carparks["night_parking"].str.upper().eq("YES")

# car_park_basement: sample values are "Y" / "N"
# converting it to boolean
carparks["is_basement"] = carparks["car_park_basement"].str.upper().eq("Y")

# validation: make sure every row with original text got a real True/False, nothing silently lost
for new_col, raw_col in [("has_night_parking", "night_parking"), ("is_basement", "car_park_basement")]:
    missing = carparks[carparks[raw_col].notna() & carparks[new_col].isna()]
    print(f"{new_col}: {len(missing)} rows failed to classify")
    if len(missing) > 0:
        print(missing[[raw_col]].drop_duplicates())

carparks.to_csv("../data/HDBCarparkInformation_new.csv", index=False)

In [ ]:
# add a carpark_id column as the primary key
# remove it first if it already exists somewhere else in the dataframe, to avoid a duplicate
if "carpark_id" in carparks.columns:
    carparks = carparks.drop(columns=["carpark_id"])

# insert it fresh as the very first column
carparks.insert(0, "carpark_id", range(1, len(carparks) + 1))

carparks.to_csv("../data/HDBCarparkInformation_new.csv", index=False)

In [ ]:
carparks.head(3)

In [ ]:
# drop x- & y-coordinates and form a new table
carparks = carparks.drop(columns=["x_coord", "y_coord"])
carparks.to_csv("../data/HDBCarparkInformation_upload.csv", index=False)

In [ ]:
print(list(carparks.columns))